# Backtests DAX 40 — ASRS · Expresso · PDH/PDL

Présentation des trois stratégies intraday sur le DAX 40 (données CSV backtesting.com 2006–2026 + API Capital.com).

| Stratégie | Signal | Buffer | Filtre |
|-----------|--------|--------|--------|
| **ASRS** | 4ème bougie 5min (09:15) | 2 pts | aucun |
| **Expresso** | Pré-ouverture (08:55) | 17 pts | range 10–55, pas vendredi, pas jan/juil/août |
| **PDH/PDL** | Haut/Bas de la veille | 5 pts | range 50–300, pas vendredi, pas jan/juil/août |

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print(f'ROOT = {ROOT}')

In [ ]:
# ── Chargement données CSV (2006-2026) ────────────────────────────────────────
raw = pd.read_csv(
    ROOT / 'data' / 'dax-5m_bk.csv', sep=';', header=None,
    names=['date', 'time', 'open', 'high', 'low', 'close', 'volume'],
)
raw['datetime'] = pd.to_datetime(raw['date'] + ' ' + raw['time'], format='%d/%m/%Y %H:%M')
raw = raw.drop(columns=['date', 'time']).set_index('datetime').sort_index()
raw = raw[raw.index >= '2006-01-01']

# ── Données Capital.com (live) ────────────────────────────────────────────────
live_raw = pd.read_csv(ROOT / 'data' / 'dax_live_5min.csv', index_col=0, parse_dates=True)
live_raw.index = pd.to_datetime(live_raw.index, utc=True).tz_convert('Europe/Berlin').tz_localize(None)
live = live_raw[~live_raw.index.duplicated()].sort_index().between_time('09:00', '17:35')

print(f'CSV  : {len(raw):,} lignes  {raw.index[0].date()} → {raw.index[-1].date()}')
print(f'Live : {len(live):,} lignes  {live.index[0].date()} → {live.index[-1].date()}')

---
## 1. ASRS — Advanced School Run Strategy

**Signal** : High/Low de la 4ème bougie 5min (09:15–09:20 CET).  
**Logique** : On place deux ordres stop (buy stop au-dessus, sell stop en-dessous). Le premier déclenché devient la position. Le stop loss = l'autre niveau d'entrée (risque = range + 4 pts). Sortie fin de journée 17:30.

In [ ]:
ENTRY_BUFFER_ASRS = 2
SIGNAL_BAR_4      = '09:15'
SIGNAL_BAR_5      = '09:20'
NARROW_THRESHOLD  = 10
EOD_ASRS          = '17:30'

# Extraction bougies signal
bars_4th = raw[raw.index.strftime('%H:%M') == SIGNAL_BAR_4].copy()
bars_4th.index = bars_4th.index.normalize()
bars_4th_by_date = bars_4th

bars_5th = raw[raw.index.strftime('%H:%M') == SIGNAL_BAR_5].copy()
bars_5th.index = bars_5th.index.normalize()
bars_5th_by_date = bars_5th

print(f'4th bar days : {len(bars_4th):,}')
print(f'5th bar days : {len(bars_5th):,}')

In [ ]:
def simulate_day_asrs(day_bars, sig_high, sig_low, bar_used=4):
    if day_bars.empty:
        return None
    el = sig_high + ENTRY_BUFFER_ASRS
    es = sig_low  - ENTRY_BUFFER_ASRS
    direction = entry_price = entry_time = stop = None

    for ts, bar in day_bars.iterrows():
        if direction:
            break
        bh, bl = bar['high'], bar['low']
        if bh >= el and bl <= es:
            direction, entry_price, stop = ('long', el, es) if bar['open'] >= el else ('short', es, el)
        elif bh >= el:
            direction, entry_price, stop = 'long',  el, es
        elif bl <= es:
            direction, entry_price, stop = 'short', es, el
        if direction:
            entry_time = ts

    if not direction:
        return None

    max_adv = max_fav = 0.0
    exit_price = exit_reason = None
    for ts, bar in day_bars[day_bars.index >= entry_time].iterrows():
        bh, bl = bar['high'], bar['low']
        if direction == 'long':
            max_adv = max(max_adv, entry_price - bl)
            max_fav = max(max_fav, bh - entry_price)
            if bl <= stop:
                exit_price, exit_reason = stop, 'stop'
                break
        else:
            max_adv = max(max_adv, bh - entry_price)
            max_fav = max(max_fav, entry_price - bl)
            if bh >= stop:
                exit_price, exit_reason = stop, 'stop'
                break

    if exit_price is None:
        exit_price, exit_reason = day_bars[day_bars.index >= entry_time].iloc[-1]['close'], 'eod'

    pnl = (exit_price - entry_price) if direction == 'long' else (entry_price - exit_price)
    return {'direction': direction, 'entry_price': round(entry_price, 2),
            'exit_price': round(exit_price, 2), 'stop': round(stop, 2),
            'pnl': round(pnl, 2), 'exit_reason': exit_reason,
            'max_adverse': round(max_adv, 2), 'max_favorable': round(max_fav, 2),
            'sig_range': round(sig_high - sig_low, 2), 'bar_used': bar_used}


def run_backtest_asrs(use_5th_fallback=False):
    trades = []
    for trade_date in bars_4th_by_date.index.unique():
        row4 = bars_4th_by_date.loc[trade_date]
        if isinstance(row4, pd.DataFrame):
            row4 = row4.iloc[0]
        sig_high, sig_low, bar_used = row4['high'], row4['low'], 4
        after = '09:20'

        if use_5th_fallback and (sig_high - sig_low) < NARROW_THRESHOLD:
            if trade_date in bars_5th_by_date.index:
                row5 = bars_5th_by_date.loc[trade_date]
                if isinstance(row5, pd.DataFrame):
                    row5 = row5.iloc[0]
                sig_high, sig_low, bar_used, after = row5['high'], row5['low'], 5, '09:25'
            else:
                continue

        day_bars = raw.loc[f"{trade_date} {after}":f"{trade_date} {EOD_ASRS}"]
        if len(day_bars) < 2:
            continue
        r = simulate_day_asrs(day_bars, sig_high, sig_low, bar_used)
        if r:
            r['trade_date'] = pd.Timestamp(trade_date)
            trades.append(r)

    return pd.DataFrame(trades).set_index('trade_date').sort_index()


asrs_results = {
    'ASRS_4th':          run_backtest_asrs(use_5th_fallback=False),
    'ASRS_4th_fallback': run_backtest_asrs(use_5th_fallback=True),
}
for k, df in asrs_results.items():
    print(f'  {k}: {len(df):,} trades')

In [ ]:
def compute_metrics(df, label=''):
    if df.empty:
        return {}
    winners, losers = df[df['pnl'] > 0], df[df['pnl'] < 0]
    gw = winners['pnl'].sum() if len(winners) else 0
    gl = abs(losers['pnl'].sum()) if len(losers) else 1e-9
    std = df['pnl'].std()
    cum = df['pnl'].cumsum()
    return {
        'label':          label,
        'n_trades':       len(df),
        'win_rate_%':     round((df['pnl'] > 0).mean() * 100, 1),
        'avg_win':        round(winners['pnl'].mean(), 1) if len(winners) else 0,
        'avg_loss':       round(losers['pnl'].mean(),  1) if len(losers)  else 0,
        'profit_factor':  round(gw / gl, 2),
        'total_pnl':      round(df['pnl'].sum(), 0),
        'sharpe':         round(df['pnl'].mean() / std * np.sqrt(252), 2) if std > 0 else 0,
        'max_dd':         round((cum - cum.cummax()).min(), 0),
    }

summary_asrs = pd.DataFrame([compute_metrics(df, k) for k, df in asrs_results.items()]).set_index('label')
print('=== ASRS — Performance Summary ===')
print(summary_asrs.to_string())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
fig.suptitle('ASRS — Equity Curves (DAX 2006–2026)', fontsize=14, fontweight='bold')

styles = [('ASRS_4th', '#1565C0', '-', 'ASRS 4th bar'),
          ('ASRS_4th_fallback', '#FF6F00', '--', 'ASRS 4th + 5th fallback')]

for ax, (key, color, ls, label) in zip(axes, styles):
    df_v = asrs_results[key]
    cum  = df_v['pnl'].cumsum()
    ax.fill_between(cum.index, cum.values, 0, where=cum.values >= 0, alpha=0.2, color='green')
    ax.fill_between(cum.index, cum.values, 0, where=cum.values  < 0, alpha=0.2, color='red')
    ax.plot(cum.index, cum.values, color=color, lw=1.8, ls=ls)
    ax.axhline(0, color='gray', lw=0.8)
    ax.set_ylabel('PnL cumulé (pts)')
    ax.grid(True, alpha=0.3)
    m = compute_metrics(df_v)
    ax.set_title(f"{label} | Trades: {m['n_trades']:,} | WR: {m['win_rate_%']}% | PF: {m['profit_factor']} | PnL: {m['total_pnl']:+,.0f} pts | MaxDD: {m['max_dd']:,.0f} pts")

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig(ROOT / 'data' / 'asrs_equity.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Expresso

**Signal** : High/Low de la bougie pré-ouverture 08:55 (ferme à 09:00 CET).  
**Logique** : Même mécanique OCO que l'ASRS mais avec un buffer plus large (±17 pts). Filtres : range 10–55 pts, pas de vendredi, pas janvier/juillet/août.

In [ ]:
ENTRY_BUFFER_EXP = 17
RANGE_MIN_EXP    = 10
RANGE_MAX_EXP    = 55
SKIP_DOW_EXP     = [4]
SKIP_MONTHS_EXP  = [1, 7, 8]
EOD_EXP          = '17:30'

bars_exp = raw[raw.index.strftime('%H:%M') == '08:55'].copy()
bars_exp.index = bars_exp.index.normalize()

bars_asrs_exp = raw[raw.index.strftime('%H:%M') == '09:15'].copy()
bars_asrs_exp.index = bars_asrs_exp.index.normalize()

print(f'Expresso bar days : {len(bars_exp):,}')
print(f'ASRS bar days     : {len(bars_asrs_exp):,}')

In [ ]:
def simulate_day_exp(day_bars, sig_high, sig_low):
    if day_bars.empty:
        return None
    el = sig_high + ENTRY_BUFFER_EXP
    es = sig_low  - ENTRY_BUFFER_EXP
    direction = entry_price = entry_time = stop = None

    for ts, bar in day_bars.iterrows():
        if direction:
            break
        bh, bl = bar['high'], bar['low']
        if bh >= el and bl <= es:
            direction, entry_price, stop = ('long', el, es) if bar['open'] >= el else ('short', es, el)
        elif bh >= el:
            direction, entry_price, stop = 'long',  el, es
        elif bl <= es:
            direction, entry_price, stop = 'short', es, el
        if direction:
            entry_time = ts

    if not direction:
        return None

    max_adv = max_fav = 0.0
    exit_price = exit_reason = None
    for ts, bar in day_bars[day_bars.index >= entry_time].iterrows():
        bh, bl = bar['high'], bar['low']
        if direction == 'long':
            max_adv = max(max_adv, entry_price - bl)
            max_fav = max(max_fav, bh - entry_price)
            if bl <= stop:
                exit_price, exit_reason = stop, 'stop'
                break
        else:
            max_adv = max(max_adv, bh - entry_price)
            max_fav = max(max_fav, entry_price - bl)
            if bh >= stop:
                exit_price, exit_reason = stop, 'stop'
                break

    if exit_price is None:
        exit_price, exit_reason = day_bars[day_bars.index >= entry_time].iloc[-1]['close'], 'eod'

    pnl = (exit_price - entry_price) if direction == 'long' else (entry_price - exit_price)
    return {'direction': direction, 'entry_price': round(entry_price, 2),
            'exit_price': round(exit_price, 2), 'stop': round(stop, 2),
            'pnl': round(pnl, 2), 'exit_reason': exit_reason,
            'max_adverse': round(max_adv, 2), 'max_favorable': round(max_fav, 2),
            'sig_range': round(sig_high - sig_low, 2)}


def run_expresso(no_friday=False, range_filter=False, skip_months=False, bars_ref=None, after_time='09:00'):
    if bars_ref is None:
        bars_ref = bars_exp
    trades = []
    for trade_date in bars_ref.index.unique():
        ts = pd.Timestamp(trade_date)
        if no_friday   and ts.dayofweek in SKIP_DOW_EXP:   continue
        if skip_months and ts.month     in SKIP_MONTHS_EXP: continue
        row = bars_ref.loc[trade_date]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        sig_high, sig_low = row['high'], row['low']
        if range_filter and not (RANGE_MIN_EXP <= sig_high - sig_low <= RANGE_MAX_EXP):
            continue
        day_bars = raw.loc[f"{trade_date} {after_time}":f"{trade_date} {EOD_EXP}"]
        if len(day_bars) < 2:
            continue
        r = simulate_day_exp(day_bars, sig_high, sig_low)
        if r:
            r['trade_date'] = ts
            trades.append(r)
    return pd.DataFrame(trades).set_index('trade_date').sort_index()


exp_results = {
    'EXP_raw':  run_expresso(),
    'EXP_best': run_expresso(no_friday=True, range_filter=True, skip_months=True),
    'ASRS_ref': run_expresso(no_friday=True, range_filter=True, skip_months=True,
                             bars_ref=bars_asrs_exp, after_time='09:20'),
}
for k, df in exp_results.items():
    print(f'  {k}: {len(df):,} trades')

In [ ]:
summary_exp = pd.DataFrame([compute_metrics(df, k) for k, df in exp_results.items()]).set_index('label')
print('=== Expresso — Performance Summary ===')
print(summary_exp.to_string())

In [ ]:
palette = {
    'EXP_raw':  ('#BDBDBD', '--', 1.2, 'Expresso sans filtre'),
    'EXP_best': ('#B71C1C', '-',  2.5, 'Expresso BEST (range+fri+mois)'),
    'ASRS_ref': ('#1A237E', '--', 2.0, 'ASRS référence (mêmes filtres)'),
}

fig, ax = plt.subplots(figsize=(15, 7))
fig.suptitle('Expresso — Equity Curves (DAX 2006–2026)', fontsize=14, fontweight='bold')

for key, (color, ls, lw, label) in palette.items():
    cum = exp_results[key]['pnl'].cumsum()
    ax.plot(cum.index, cum.values, color=color, lw=lw, ls=ls, label=label)
    ax.annotate(f"{cum.iloc[-1]:+,.0f}", xy=(cum.index[-1], cum.iloc[-1]),
                xytext=(6, 0), textcoords='offset points', color=color, fontsize=9, fontweight='bold')

ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('PnL cumulé (pts)')
ax.set_xlabel('Date')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'data' / 'expresso_equity.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. PDH/PDL — Previous Day High / Low

**Signal** : High et Low de la journée précédente (session 09:00–17:35 CET).  
**Logique** : Achat stop au-dessus du PDH+5 pts, vente stop en-dessous du PDL−5 pts. Stop loss = l'autre niveau. Filtre : range journalier 50–300 pts, pas vendredi, pas jan/juil/août.  
**Source** : données API Capital.com (mid bid/ask).

In [ ]:
SKIP_DOW_PDH    = [4]
SKIP_MONTHS_PDH = [1, 7, 8]
BUFFER_PDH      = 5
RANGE_MIN_PDH   = 50
RANGE_MAX_PDH   = 300

def simulate_pdhl(day_bars, sig_high, sig_low, buffer=5):
    if day_bars.empty:
        return None
    el = sig_high + buffer
    es = sig_low  - buffer
    direction = entry_price = entry_time = stop = None

    for ts, bar in day_bars.iterrows():
        if direction:
            break
        bh, bl = bar['high'], bar['low']
        if bh >= el and bl <= es:
            direction, entry_price, stop = ('long', el, es) if bar['open'] >= el else ('short', es, el)
        elif bh >= el:
            direction, entry_price, stop = 'long',  el, es
        elif bl <= es:
            direction, entry_price, stop = 'short', es, el
        if direction:
            entry_time = ts

    if not direction:
        return None

    exit_price = None
    for ts, bar in day_bars[day_bars.index >= entry_time].iterrows():
        if direction == 'long':
            if bar['low'] <= stop:
                exit_price = stop
                break
        else:
            if bar['high'] >= stop:
                exit_price = stop
                break

    if exit_price is None:
        exit_price = day_bars[day_bars.index >= entry_time].iloc[-1]['close']

    pnl = (exit_price - entry_price) if direction == 'long' else (entry_price - exit_price)
    return {'direction': direction, 'entry': round(entry_price, 1),
            'exit': round(exit_price, 1), 'stop': round(stop, 1),
            'pdh': round(sig_high, 1), 'pdl': round(sig_low, 1),
            'pnl': round(pnl, 2)}


def run_pdhl(df_5min, label):
    dates  = sorted(set(df_5min.index.date))
    trades = []
    for i, d in enumerate(dates):
        if i == 0:
            continue
        ts = pd.Timestamp(d)
        if ts.dayofweek in SKIP_DOW_PDH or ts.month in SKIP_MONTHS_PDH:
            continue
        prev_b = df_5min.loc[str(dates[i - 1])]
        if len(prev_b) < 2:
            continue
        sh, sl = prev_b['high'].max(), prev_b['low'].min()
        if not (RANGE_MIN_PDH <= sh - sl <= RANGE_MAX_PDH):
            continue
        day_b = df_5min.loc[str(d)]
        if len(day_b) < 2:
            continue
        r = simulate_pdhl(day_b, sh, sl, BUFFER_PDH)
        if r:
            r['date'] = ts
            trades.append(r)

    if not trades:
        print(f'{label} : aucun trade')
        return pd.DataFrame()

    df_t = pd.DataFrame(trades).set_index('date')
    pnl  = df_t['pnl']
    wins = pnl[pnl > 0]; losses = pnl[pnl < 0]
    pf   = wins.sum() / abs(losses.sum()) if len(losses) else float('inf')
    print(f'{label}')
    print(f'  Trades : {len(pnl):,}  |  WR : {(pnl>0).mean()*100:.0f}%  |  PF : {pf:.2f}  |  PnL : {pnl.sum():+.1f} pts')
    return df_t


print('=== PDH/PDL ===')
trades_live = run_pdhl(live, 'Capital.com (live)')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('PDH/PDL — Equity Curve (Capital.com)', fontsize=13, fontweight='bold')

cum    = trades_live['pnl'].cumsum()
colors = ['green' if p > 0 else 'red' for p in trades_live['pnl']]

ax2 = ax.twinx()
ax.bar(range(len(trades_live)), trades_live['pnl'], color=colors, alpha=0.6)
ax2.plot(range(len(trades_live)), cum, color='#1565C0', lw=2, label='PnL cumulé')
ax2.axhline(0, color='black', lw=0.8)

ax.set_xlabel('Trade #')
ax.set_ylabel('PnL par trade (pts)')
ax2.set_ylabel('PnL cumulé (pts)')
ax2.legend(loc='upper left')
ax.grid(True, alpha=0.3)

m = compute_metrics(trades_live.rename(columns={'pnl': 'pnl'}))
ax.set_title(f"Capital.com | Trades: {len(trades_live)} | WR: {(trades_live['pnl']>0).mean()*100:.0f}% | PnL: {trades_live['pnl'].sum():+.1f} pts")

plt.tight_layout()
plt.savefig(ROOT / 'data' / 'pdhl_equity.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Comparaison des 3 stratégies

> PDH/PDL uniquement sur la période Capital.com (données live). ASRS et Expresso sur 20 ans CSV.

In [ ]:
print(f"{'Stratégie':<20} {'Trades':>8} {'WR':>6} {'PF':>6} {'PnL (pts)':>12} {'MaxDD':>8}")
print('-' * 65)

for label, df in [
    ('ASRS 4th bar',    asrs_results['ASRS_4th']),
    ('ASRS + fallback', asrs_results['ASRS_4th_fallback']),
    ('Expresso BEST',   exp_results['EXP_best']),
    ('PDH/PDL live',    trades_live),
]:
    pnl = df['pnl']
    wins   = pnl[pnl > 0].sum()
    losses = abs(pnl[pnl < 0].sum())
    pf     = wins / losses if losses else float('inf')
    cum    = pnl.cumsum()
    max_dd = (cum - cum.cummax()).min()
    print(f"{label:<20} {len(pnl):>8,} {(pnl>0).mean()*100:>5.0f}% {pf:>6.2f} {pnl.sum():>+12.0f} {max_dd:>8.0f}")